# Sessions & Memory

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) OpenAI Agents SDK roadmap.

You will learn how to use `SQLiteSession` for persistent conversation memory, manage session history with `get_items`, `add_items`, and `clear_session`, and control history size with `SessionSettings`.

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Create an Agent with SQLiteSession

Use `SQLiteSession` to persist conversation history across runs. Each session is identified by a unique `session_id`.

In [ ]:
from agents import Agent, Runner, SQLiteSession

agent = Agent(
    name="Memory Agent",
    instructions="You are a helpful assistant with memory of past conversations.",
)

session = SQLiteSession("conversations.db", session_id="user_123")

result = await Runner.run(agent, "My name is Alice and I love Python.", session=session)
print(result.final_output)

## 4. Continuing a Conversation

Use the same `session_id` to resume where you left off. The SDK automatically replays previous history.

In [ ]:
session = SQLiteSession("conversations.db", session_id="user_123")

result = await Runner.run(agent, "What's my name and what do I love?", session=session)
print(result.final_output)

## 5. Inspecting Session Contents

Use `get_items()` to view what's stored in a session.

In [ ]:
session = SQLiteSession("conversations.db", session_id="user_123")

items = session.get_items()
for item in items:
    print(item)

## 6. Adding Items Manually

Pre-populate a session with items using `add_items`.

In [ ]:
session = SQLiteSession("conversations.db", session_id="new_user")

session.add_items([
    {"role": "user", "content": "I prefer dark mode."},
    {"role": "assistant", "content": "Noted! I'll remember you prefer dark mode."},
])

result = await Runner.run(agent, "What's my preference?", session=session)
print(result.final_output)

## 7. Clearing a Session

Reset a session to start fresh with `clear_session()`.

In [ ]:
session = SQLiteSession("conversations.db", session_id="user_123")
session.clear_session()

result = await Runner.run(agent, "Do you remember my name?", session=session)
print(result.final_output)

## 8. Controlling History with SessionSettings

Use `SessionSettings(limit=N)` to cap how many items are replayed, keeping token usage manageable.

In [ ]:
from agents import SessionSettings

session = SQLiteSession(
    "conversations.db",
    session_id="limited_user",
    settings=SessionSettings(limit=10),
)

result = await Runner.run(agent, "Remember: my favorite color is blue.", session=session)
print(result.final_output)

## 9. Multiple Sessions

Different `session_id` values create independent conversation histories.

In [ ]:
session_alice = SQLiteSession("conversations.db", session_id="alice")
session_bob = SQLiteSession("conversations.db", session_id="bob")

await Runner.run(agent, "I'm Alice, I like cats.", session=session_alice)
await Runner.run(agent, "I'm Bob, I like dogs.", session=session_bob)

result = await Runner.run(agent, "What do I like?", session=session_alice)
print(f"Alice's session: {result.final_output}")

result = await Runner.run(agent, "What do I like?", session=session_bob)
print(f"Bob's session: {result.final_output}")

## Key Takeaways

- `SQLiteSession` provides persistent memory across multiple `Runner.run` calls
- Each session is identified by a unique `session_id` — different IDs create independent histories
- Use `get_items` and `add_items` to inspect or pre-populate session data
- `clear_session` resets the conversation history
- `SessionSettings(limit=N)` controls how much history is replayed, keeping token usage in check